# Modèle Prédictif — Meta Score T+1 mois

**Objectif :** prédire le `meta_score` d'un archetype le mois prochain.

**Méthode :**
- Features temporelles : lags T-1/T-2/T-3, momentum (delta), accélération, rolling mean
- Features de rang : percentile dans le mois (invariant au niveau global)
- Walk-forward cross-validation : fenêtre glissante de 9 mois (évite le leakage + distribution shift)
- Ensemble : 70% stabilité (naïf) + 30% modèle Ridge → meilleure ρ de Spearman
- Métrique principale : **Spearman ρ** (corrélation de rang) — plus robuste que R² ici

**Découverte clé :** la méta est "sticky" — Spearman ρ naïf = +0.508 en 2026.
Le modèle apporte un signal marginal sur les archetypes en transition.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

con = sqlite3.connect('../data/yugioh.db')
ms = pd.read_sql("""
    SELECT month, archetype, share, avg_placement, meta_score
    FROM meta_scores ORDER BY month, archetype
""", con)
ms['month'] = pd.to_datetime(ms['month'])
print(f'meta_scores : {len(ms):,} lignes, {ms["month"].nunique()} mois, {ms["archetype"].nunique()} archetypes')

## 1. Construction des features

In [ ]:
## 1. Features statiques (banlist, trend)

## 2. Features temporelles — lags, momentum, rang

g = ms.sort_values(['archetype', 'month']).groupby('archetype')

# Lags T-1, T-2, T-3
for col in ['meta_score', 'share', 'avg_placement']:
    for lag in [1, 2, 3]:
        ms[f'{col}_t{lag}'] = g[col].shift(lag)

# Momentum : delta 1m, 2m + accélération
ms['delta_1m'] = ms['meta_score'] - ms['meta_score_t1']
ms['delta_2m'] = ms['meta_score'] - ms['meta_score_t2']
ms['accel']    = ms['delta_1m'] - (ms['meta_score_t1'] - ms['meta_score_t2'])

# Rolling mean 3m (sur T-1, T-2, T-3 — pas de leakage)
ms['roll_mean_3m'] = g['meta_score'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=2).mean()
)

# Rang percentile dans le mois (invariant au niveau global du meta_score)
ms['rank_month'] = ms.groupby('month')['meta_score'].rank(pct=True)
ms['rank_t1']    = ms.groupby('month')['meta_score_t1'].rank(pct=True)

# Target : meta_score T+1 et delta T+1
ms['meta_score_next'] = g['meta_score'].shift(-1)
ms['month_next']      = g['month'].shift(-1)
ms['delta_next']      = ms['meta_score_next'] - ms['meta_score']

# Filtrer : T+1 doit être le mois calendaire suivant
ms_clean = ms.dropna(subset=['meta_score_next', 'meta_score_t1', 'delta_next'])
ms_clean = ms_clean[(ms_clean['month_next'] - ms_clean['month']).dt.days.between(25, 35)]

dataset = ms_clean.merge(static, on='archetype', how='left').fillna(0)

FEATURE_COLS = [
    # Niveaux absolus
    'meta_score', 'meta_score_t1', 'meta_score_t2', 'meta_score_t3',
    # Momentum
    'delta_1m', 'delta_2m', 'accel', 'roll_mean_3m',
    # Part et placement
    'share', 'share_t1', 'avg_placement', 'avg_placement_t1',
    # Rang relatif dans le mois (stationnaire)
    'rank_month', 'rank_t1',
    # Banlist
    'trend_ratio', 'n_banned', 'n_limited',
]

print(f'Dataset : {len(dataset)} exemples, {len(FEATURE_COLS)} features')
print(f'Mois couverts : {dataset["month"].min().date()} → {dataset["month"].max().date()}')
print(f'\nDistrib delta_next : mean={dataset["delta_next"].mean():+.4f}  std={dataset["delta_next"].std():.4f}')

## 3. Walk-Forward Cross-Validation (fenêtre 9 mois)

WINDOW = 9   # fenêtre glissante en mois
W_MODEL = 0.3  # poids modèle dans l'ensemble (70% naïf + 30% modèle = meilleur en WF-CV)

months = sorted(dataset['month'].unique())
wf_results = []

for i in range(WINDOW, len(months)):
    train_months = months[max(0, i - WINDOW):i]
    test_month   = months[i]

    train_mask = dataset['month'].isin(train_months)
    test_mask  = dataset['month'] == test_month

    X_tr = dataset.loc[train_mask, FEATURE_COLS].values
    X_te = dataset.loc[test_mask,  FEATURE_COLS].values
    y_tr = dataset.loc[train_mask, 'delta_next'].values
    y_te = dataset.loc[test_mask,  'meta_score_next'].values
    cur  = dataset.loc[test_mask,  'meta_score'].values

    if len(X_te) < 3:
        continue

    scaler = StandardScaler()
    model  = Ridge(alpha=50.0)
    model.fit(scaler.fit_transform(X_tr), y_tr)
    pred_model   = cur + model.predict(scaler.transform(X_te))
    pred_blended = W_MODEL * pred_model + (1 - W_MODEL) * cur

    rho_model,   _ = spearmanr(y_te, pred_model)
    rho_blended, _ = spearmanr(y_te, pred_blended)
    rho_naive,   _ = spearmanr(y_te, cur)
    r2_blended     = r2_score(y_te, pred_blended)

    wf_results.append({
        'month': test_month, 'n_test': len(y_te),
        'rho_model': rho_model, 'rho_blended': rho_blended, 'rho_naive': rho_naive,
        'r2_blended': r2_blended,
        'beats_naive': rho_blended > rho_naive,
    })

wf_df = pd.DataFrame(wf_results)

print(f'Walk-Forward CV — {len(wf_df)} mois testés (fenêtre={WINDOW}m)\n')
print(f'{"":25s}  {"ρ mean":>8}  {"ρ median":>9}')
print(f'  Naïf "no change"         {wf_df["rho_naive"].mean():+.3f}     {wf_df["rho_naive"].median():+.3f}')
print(f'  Modèle Ridge (δ)         {wf_df["rho_model"].mean():+.3f}     {wf_df["rho_model"].median():+.3f}')
print(f'  Ensemble 30/70           {wf_df["rho_blended"].mean():+.3f}     {wf_df["rho_blended"].median():+.3f}')

won = wf_df['beats_naive'].sum()
print(f'\nEnsemble bat naïf : {won}/{len(wf_df)} mois ({100*won/len(wf_df):.0f}%)')

recent = wf_df[wf_df['month'] >= '2026-01-01']
if len(recent):
    print(f'\nMois 2026+ uniquement ({len(recent)}) :')
    print(f'  Naïf       ρ = {recent["rho_naive"].mean():+.3f}')
    print(f'  Ensemble   ρ = {recent["rho_blended"].mean():+.3f}')

print('\nDétail par mois :')
for _, r in wf_df.iterrows():
    marker = "✓" if r['beats_naive'] else "✗"
    print(f'  {r["month"].strftime("%Y-%m")} n={r["n_test"]:2.0f}  '
          f'ρ_blend={r["rho_blended"]:+.3f}  ρ_naive={r["rho_naive"]:+.3f}  {marker}')

## 4. Feature importance

# Entraîner sur tout le dataset pour la feature importance
scaler_full = StandardScaler()
rf_full = RandomForestRegressor(n_estimators=300, max_depth=4, min_samples_leaf=2, random_state=42)
rf_full.fit(
    scaler_full.fit_transform(dataset[FEATURE_COLS].values),
    dataset['delta_next'].values
)
importances = pd.Series(rf_full.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

print('Feature importance — RF entraîné sur tout le dataset (target = delta_next) :')
for feat, imp in importances.items():
    bar = '█' * int(imp * 50)
    print(f'  {feat:22s}  {imp:.3f}  {bar}')

## 5. Prédictions mois prochain (archetypes actifs récemment)

import datetime

# Entraîner sur la fenêtre glissante la plus récente
train_months = months[-WINDOW:]
train_mask   = dataset['month'].isin(train_months)
X_tr_final   = dataset.loc[train_mask, FEATURE_COLS].values
y_tr_final   = dataset.loc[train_mask, 'delta_next'].values

scaler_pred = StandardScaler()
model_pred  = Ridge(alpha=50.0)
model_pred.fit(scaler_pred.fit_transform(X_tr_final), y_tr_final)

# État le plus récent de chaque archetype — filtrer à < 4 mois (actifs récemment)
RECENCY_MONTHS = 4
cutoff = dataset['month'].max() - pd.DateOffset(months=RECENCY_MONTHS)
last_ms = (ms_clean[ms_clean['month'] >= cutoff]
           .sort_values('month')
           .groupby('archetype')
           .last()
           .reset_index())

last_ms = last_ms.merge(static, on='archetype', how='left').fillna(0)

for col in FEATURE_COLS:
    if col not in last_ms.columns:
        last_ms[col] = 0.0

X_pred_final = last_ms[FEATURE_COLS].fillna(0).values
deltas       = model_pred.predict(scaler_pred.transform(X_pred_final))

last_ms['pred_delta']      = deltas
last_ms['pred_meta_score'] = (last_ms['meta_score'] + W_MODEL * deltas).clip(lower=0)
last_ms['pred_direction']  = np.where(deltas > 0.01, '↑', np.where(deltas < -0.01, '↓', '→'))
last_ms['data_month']      = last_ms['month'].dt.strftime('%Y-%m')

preds = last_ms[['archetype','data_month','meta_score','pred_delta','pred_meta_score','pred_direction']]
preds = preds.sort_values('pred_meta_score', ascending=False)

print(f'=== PRÉDICTIONS — TOP 20 ARCHETYPES (entraîné sur {train_months[0].strftime("%Y-%m")}→{train_months[-1].strftime("%Y-%m")}) ===\n')
print(preds.head(20).to_string(index=False, float_format='{:.4f}'.format))

print('\n=== ARCHETYPES EN HAUSSE (delta > 0.01) ===')
print(preds[preds['pred_direction']=='↑'][['archetype','meta_score','pred_delta','pred_meta_score']].head(10).to_string(index=False, float_format='{:.4f}'.format))

# Sauvegarder
con2 = sqlite3.connect('../data/yugioh.db')
con2.execute("DROP TABLE IF EXISTS meta_predictions")
con2.execute("""CREATE TABLE meta_predictions (
    archetype TEXT, data_month TEXT, meta_score_current REAL,
    pred_delta REAL, pred_meta_score REAL, pred_direction TEXT, computed_at TEXT
)""")
out = preds.copy()
out.columns = ['archetype','data_month','meta_score_current','pred_delta','pred_meta_score','pred_direction']
out['computed_at'] = datetime.date.today().isoformat()
out.to_sql('meta_predictions', con2, if_exists='append', index=False)
con2.commit(); con2.close()
print(f'\nSauvegardé : {len(out)} prédictions dans meta_predictions')
print(f'Note : Spearman ρ validé en walk-forward CV = +{wf_df["rho_blended"].mean():.3f} (moyen), +{recent["rho_blended"].mean():.3f} (2026)')

In [ ]:
# Prendre le dernier mois connu pour chaque archetype
last_known = (ms.sort_values('month')
              .groupby('archetype')
              .last()
              .reset_index())

last_known = last_known.merge(features, on='archetype', how='left').fillna(0)

X_future = last_known[FEATURE_COLS].values
X_future_s = scaler.transform(X_future)

# Utiliser le meilleur modèle (Gradient Boosting généralement)
best_model_name = max(results, key=lambda k: results[k]['r2'])
best_model = results[best_model_name]['model']

if best_model_name == 'Ridge Regression':
    preds_future = best_model.predict(X_future_s)
else:
    preds_future = best_model.predict(X_future)

last_known['predicted_next_month'] = preds_future
last_known['delta'] = last_known['predicted_next_month'] - last_known['meta_score']

predictions = last_known[['archetype', 'meta_score', 'predicted_next_month', 'delta']].sort_values('predicted_next_month', ascending=False)

print(f'Prédictions mois prochain (modèle : {best_model_name}) :')
print()
print('=== ARCHETYPES QUI VONT MONTER ===')
print(predictions[predictions['delta'] > 0].head(10).to_string(index=False, float_format='{:.3f}'.format))
print()
print('=== ARCHETYPES QUI VONT BAISSER ===')
print(predictions[predictions['delta'] < 0].sort_values('delta').head(10).to_string(index=False, float_format='{:.3f}'.format))